In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import average_precision_score
from sklearn.model_selection import StratifiedKFold
from pyod.models.deep_svdd import DeepSVDD
import sys
import scrapbook as sb

sys.path.append('..')
from utils import reshape_to_numpy, interpolate_missing_values, denoise_data, compute_spectrograms, time_avg_pooling

In [2]:
seed = 1
sampling_rate = 10
nperseg=64
noverlap=32
accel_cutoff = 0.4
accel_order = 30
gyro_cutoff = 0.6
gyro_order = 20
batch_size = 32
epochs = 100
hidden_neurons = [128, 64, 32]
n_splits = 5

In [3]:
# Parameters
seed = 22


In [4]:
rng = np.random.RandomState(seed)

In [5]:
data = pd.read_parquet("../data/GBG500.parquet")
data

,ride_id,time_index,ax,ay,az,rx,ry,rz
0,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,0,-2.512796,-9.385012,-1.053078,-0.009155,0.009155,-0.201416
1,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,100,-2.491268,-9.385012,-1.079390,-0.036621,0.027465,-0.155639
2,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,200,-2.534324,-9.382022,-1.030952,0.036621,0.027465,-0.073242
3,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,300,-2.488278,-9.383218,-1.091948,-0.045776,0.036621,-0.183105
4,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,400,-2.483494,-9.382022,-1.100320,-0.045776,0.027465,-0.192260
...,...,...,...,...,...,...,...,...
1529542,ffb6f2f5dc6ba768d88dc0bc1af4cad235526aa9e24168...,241300,0.459862,-9.140430,-3.318900,1.556396,-0.091552,0.274658
1529543,ffb6f2f5dc6ba768d88dc0bc1af4cad235526aa9e24168...,241400,0.455676,-9.140430,-3.321292,1.583861,-0.119018,0.274658
1529544,ffb6f2f5dc6ba768d88dc0bc1af4cad235526aa9e24168...,241500,0.459862,-9.139832,-3.317704,1.583861,-0.109863,0.274658
1529545,ffb6f2f5dc6ba768d88dc0bc1af4cad235526aa9e24168...,241600,0.456274,-9.142224,-3.321292,1.574707,-0.137329,0.283813


In [6]:
labels = pd.read_csv("../data/GBG500_labels.csv")
labels.columns = labels.columns.str.lower()
ride_order_df = pd.DataFrame({"ride_id": data["ride_id"].unique()})
labels_sorted = ride_order_df.merge(
    labels,
    on="ride_id",
    how="left"
)
labels_sorted

,ride_id,label
0,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,Safe
1,00d223cb7aecc9c0cc5871e32a6c45027a068eba07c572...,Reckless
2,02288a4aeca044203394e982e76b87818021dea2b34df9...,Safe
3,0236bdcf13d473ea24d97f5eaeea459f257dffac005694...,Bad weather
4,0238e5dd85143f1b6c59b200bc32f14361b8fc84c51a87...,Safe
...,...,...
495,fe5d7f84d692bbccad8bd566a3ad5c3108fa9f90acd671...,Safe
496,fe67169632306d4668b2affedef510df2cb43e2fb41220...,Safe
497,fee44667fdc8ab4995c702fc3bd36178de0b1ca2c64687...,Safe
498,ff79b2e945b93c2a5efc3a06365267b3a160e7849ba57d...,Safe


In [7]:
data_np = reshape_to_numpy(
    data,
    features = ["ax", "ay", "az", "rx", "ry", "rz"],
    max_timestamps = 4800
)

data_np_clean = interpolate_missing_values(
    data_np,
    method='linear',
    limit=None
)

data_np_clean = denoise_data(
    data=data_np_clean,
    accel_indices=[0, 1, 2],
    gyro_indices=[3, 4, 5],
    accel_cutoff=accel_cutoff,
    accel_order=accel_order,
    gyro_cutoff=gyro_cutoff,
    gyro_order=gyro_order,
)

In [8]:
# Compute spectrograms for all rides
spectrograms_array = compute_spectrograms(data_np_clean, sampling_rate, nperseg, noverlap)
spectrograms_array.shape

(500, 149, 6, 33)

In [9]:
# Aggregate spectrograms using time-averaged pooling
X_feat = time_avg_pooling(spectrograms_array)
X_feat.shape

(500, 198)

In [10]:
# 5-fold stratified CV with Deep SVDD
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=rng.randint(1000))
y_true = (labels_sorted['label'] == 'Reckless').astype(int).values
anomaly_scores = np.full(len(y_true), np.nan)

for fold, (train_idx, test_idx) in enumerate(skf.split(X_feat, y_true)):
    sc = StandardScaler()
    X_train = sc.fit_transform(X_feat[train_idx])
    X_test = sc.transform(X_feat[test_idx])

    np.random.seed(rng.randint(1000))
    model = DeepSVDD(
        n_features=X_train.shape[1],
        hidden_neurons=hidden_neurons,
        epochs=epochs,
        batch_size=batch_size,
        random_state=rng.randint(1000),
    )
    model.fit(X_train)
    anomaly_scores[test_idx] = model.decision_function(X_test)
    print(f"Fold {fold+1}/{n_splits} done")

Epoch 1/100, Loss: 2.014463283121586
Epoch 2/100, Loss: 2.0464963987469673
Epoch 3/100, Loss: 1.8609494641423225
Epoch 4/100, Loss: 1.943739265203476
Epoch 5/100, Loss: 1.9864753037691116
Epoch 6/100, Loss: 1.9140448048710823
Epoch 7/100, Loss: 2.0219075679779053
Epoch 8/100, Loss: 2.042791113257408
Epoch 9/100, Loss: 1.7865092009305954
Epoch 10/100, Loss: 2.0305039659142494
Epoch 11/100, Loss: 1.8495717495679855
Epoch 12/100, Loss: 1.8840836212038994
Epoch 13/100, Loss: 1.9806364551186562
Epoch 14/100, Loss: 2.1691958010196686
Epoch 15/100, Loss: 2.0472176149487495


Epoch 16/100, Loss: 2.0177528262138367
Epoch 17/100, Loss: 2.009452097117901
Epoch 18/100, Loss: 2.06561391800642
Epoch 19/100, Loss: 2.04171671718359
Epoch 20/100, Loss: 2.047280788421631
Epoch 21/100, Loss: 2.0547041594982147
Epoch 22/100, Loss: 2.0352121740579605
Epoch 23/100, Loss: 2.1120061799883842
Epoch 24/100, Loss: 1.9081853926181793
Epoch 25/100, Loss: 1.919365182518959
Epoch 26/100, Loss: 1.867351621389389
Epoch 27/100, Loss: 2.1787073761224747
Epoch 28/100, Loss: 2.0563204288482666
Epoch 29/100, Loss: 2.135510064661503
Epoch 30/100, Loss: 1.9587292149662971


Epoch 31/100, Loss: 2.195937991142273
Epoch 32/100, Loss: 1.9044745340943336
Epoch 33/100, Loss: 1.8968779668211937
Epoch 34/100, Loss: 1.9668779373168945
Epoch 35/100, Loss: 1.9298179298639297
Epoch 36/100, Loss: 2.1467130556702614
Epoch 37/100, Loss: 1.985268473625183
Epoch 38/100, Loss: 1.9676236435770988
Epoch 39/100, Loss: 1.9435829259455204
Epoch 40/100, Loss: 1.9920622557401657
Epoch 41/100, Loss: 1.9712084904313087
Epoch 42/100, Loss: 2.0855205431580544
Epoch 43/100, Loss: 2.4999751299619675
Epoch 44/100, Loss: 1.971714735031128
Epoch 45/100, Loss: 2.063270553946495


Epoch 46/100, Loss: 2.2719646468758583
Epoch 47/100, Loss: 2.0559543073177338
Epoch 48/100, Loss: 2.0322984233498573
Epoch 49/100, Loss: 2.1584637612104416
Epoch 50/100, Loss: 1.9719240963459015
Epoch 51/100, Loss: 2.1604555547237396
Epoch 52/100, Loss: 1.8601328432559967
Epoch 53/100, Loss: 2.025032326579094
Epoch 54/100, Loss: 1.9246907085180283
Epoch 55/100, Loss: 2.020775094628334
Epoch 56/100, Loss: 2.0600955113768578
Epoch 57/100, Loss: 2.047623112797737
Epoch 58/100, Loss: 2.1379852443933487
Epoch 59/100, Loss: 2.2802829444408417
Epoch 60/100, Loss: 1.7758303955197334


Epoch 61/100, Loss: 1.9240906611084938
Epoch 62/100, Loss: 2.0611970126628876
Epoch 63/100, Loss: 1.9975990802049637
Epoch 64/100, Loss: 2.077221356332302
Epoch 65/100, Loss: 2.0694601610302925
Epoch 66/100, Loss: 2.0109490007162094
Epoch 67/100, Loss: 2.0593843534588814
Epoch 68/100, Loss: 2.3480141162872314
Epoch 69/100, Loss: 2.0450645238161087
Epoch 70/100, Loss: 1.938174158334732
Epoch 71/100, Loss: 1.872788280248642
Epoch 72/100, Loss: 2.0089920088648796
Epoch 73/100, Loss: 2.0409997180104256
Epoch 74/100, Loss: 2.02852813154459
Epoch 75/100, Loss: 2.0291611552238464


...

Epoch 35/100, Loss: 1.7682108283042908
Epoch 36/100, Loss: 1.8480940833687782
Epoch 37/100, Loss: 1.7883743718266487
Epoch 38/100, Loss: 1.8154473900794983
Epoch 39/100, Loss: 1.7524662539362907
Epoch 40/100, Loss: 1.7614002227783203
Epoch 41/100, Loss: 1.8456996157765388
Epoch 42/100, Loss: 1.8309847488999367
Epoch 43/100, Loss: 1.7739462405443192
Epoch 44/100, Loss: 1.8482829257845879


Epoch 45/100, Loss: 1.7987365052103996
Epoch 46/100, Loss: 1.8673278540372849
Epoch 47/100, Loss: 1.7663814723491669
Epoch 48/100, Loss: 1.7571901232004166
Epoch 49/100, Loss: 1.8431956395506859
Epoch 50/100, Loss: 1.8376993238925934
Epoch 51/100, Loss: 1.7481935247778893
Epoch 52/100, Loss: 1.8237242698669434
Epoch 53/100, Loss: 1.8252007514238358
Epoch 54/100, Loss: 1.7692571803927422
Epoch 55/100, Loss: 1.767293781042099
Epoch 56/100, Loss: 1.7483011484146118
Epoch 57/100, Loss: 1.7997987046837807


Epoch 58/100, Loss: 1.7765434607863426
Epoch 59/100, Loss: 1.7858798503875732
Epoch 60/100, Loss: 1.7909551858901978
Epoch 61/100, Loss: 1.7551319748163223
Epoch 62/100, Loss: 1.8044121786952019
Epoch 63/100, Loss: 1.7839037626981735
Epoch 64/100, Loss: 1.8091571182012558
Epoch 65/100, Loss: 1.784279614686966
Epoch 66/100, Loss: 1.7732832729816437
Epoch 67/100, Loss: 1.7976123318076134
Epoch 68/100, Loss: 1.822560541331768
Epoch 69/100, Loss: 1.8472137749195099
Epoch 70/100, Loss: 1.8214752152562141
Epoch 71/100, Loss: 1.799584448337555
Epoch 72/100, Loss: 1.8296763598918915


Epoch 73/100, Loss: 1.8010955303907394
Epoch 74/100, Loss: 1.7855473756790161
Epoch 75/100, Loss: 1.8471587747335434
Epoch 76/100, Loss: 1.8313969299197197
Epoch 77/100, Loss: 1.9022913724184036
Epoch 78/100, Loss: 1.8439863100647926
Epoch 79/100, Loss: 1.878814123570919
Epoch 80/100, Loss: 1.802818924188614
Epoch 81/100, Loss: 1.8004687130451202
Epoch 82/100, Loss: 1.7656378895044327
Epoch 83/100, Loss: 1.7954719364643097
Epoch 84/100, Loss: 1.868569903075695
Epoch 85/100, Loss: 1.8493643775582314
Epoch 86/100, Loss: 1.732563130557537
Epoch 87/100, Loss: 1.7666740417480469
Epoch 88/100, Loss: 1.877040557563305
Epoch 89/100, Loss: 1.8865623250603676
Epoch 90/100, Loss: 1.8093374148011208


Epoch 91/100, Loss: 1.8210698664188385
Epoch 92/100, Loss: 1.8492885380983353
Epoch 93/100, Loss: 1.9215306788682938
Epoch 94/100, Loss: 1.828782320022583
Epoch 95/100, Loss: 1.8695293888449669
Epoch 96/100, Loss: 1.8217668682336807
Epoch 97/100, Loss: 1.7742822393774986
Epoch 98/100, Loss: 1.8419889137148857
Epoch 99/100, Loss: 1.921316459774971
Epoch 100/100, Loss: 1.896826945245266
Fold 5/5 done


In [11]:
ap = average_precision_score(y_true, anomaly_scores)
print(f"Deep SVDD AP (5-fold CV) = {ap:.4f}")
sb.glue("GBG500_ap_spectral_deep_svdd", float(ap))

Deep SVDD AP (5-fold CV) = 0.6561
